In [1]:
!pwd

/Users/anshulchiranth/Desktop/Strike/Voice Experiments


In [2]:
import numpy as np

In [3]:
from pathlib import Path
base_dir = Path.cwd()

#Set up Various Directories
#Julian and George Raw Ground Truths Live in "Raw Ground Truths"
#They will go through processing and move to Processed Ground Truths, then prepare_audio_for_embedding() method and move to "To Be Embedded Ground Truths"
#Transaction Clips live in "Processed Transaction Clips". They only need to go through prepare_audio_for_embedding() method and move to "To Be Embedded Transactions


raw_truth_dir = base_dir / "Raw Ground Truths"
transaction_dir = base_dir / "Processed Transaction Clips"

processed_dir = base_dir / "Processed Ground Truths"
output_dir = base_dir / "To Be Embedded Ground Truths"
to_embed_dir = base_dir / "To Be Embedded Transactions"

processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents = True, exist_ok = True)
to_embed_dir.mkdir(parents = True, exist_ok=True)

In [4]:
#Import Methods from utils.py
from utils import prepare_audio_for_embedding, decode_audio, preprocess_audio, save_audio

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [5]:
#Preprocessing Steps for Raw Ground Truths
#Pass through methods decode_audio(), preprocess_audio(), and save_audio()
#decode_audio() and preprocess_audio() combine to apply bandpass, AGC and limiter (in that order)
#save_audio() will save ground truths to "Processed Ground Truths"
from tqdm import tqdm

raw_input_files = [f for f in raw_truth_dir.rglob("*")]
print(f"Found {len(raw_input_files)} raw input files.")

processed_audio_samples = []

for i, file_path in enumerate(raw_input_files):
    file_name = str(raw_input_files[i])[len(str(raw_truth_dir))+1:]
    audio, sr = decode_audio(file_path)
    processed_audio_sample = preprocess_audio(audio, sr)
    output_path = save_audio(audio, sr, processed_dir / f"Processed_{file_name}")
    print(f"Saved file to {output_path}")

Found 2 raw input files.
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_George Robbins.wav
Saved file to /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Processed Ground Truths/Processed_Julian Carmona-Munoz.wav


In [6]:
#Get Preprocessed Ground Truths Ready for Embedding
#prepare_audio_for_embedding() method will set frame rate of the .wav to 16kHz
processed_files = [f for f in processed_dir.rglob("*") if f.is_file()]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, output_dir)

Found 2 audio files


100%|██████████| 2/2 [00:00<00:00, 659.38it/s]


In [7]:
#Get Preprocessed Transaction Clips Ready for Embedding
processed_files = [f for f in transaction_dir.rglob("*") if f.is_file()]

print(f"Found {len(processed_files)} audio files")

for processed_file in tqdm(processed_files):
    processed_path = prepare_audio_for_embedding(processed_file, to_embed_dir)

Found 67 audio files


100%|██████████| 67/67 [00:00<00:00, 1687.02it/s]


In [8]:
#Standard Code to set up TitaNet
#Set to eval() mode for future embedding
import torch
import nemo.collections.asr as nemo_asr

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

# Pretrained TitaNet Large speaker verification model
speaker_model = nemo_asr.models.EncDecSpeakerLabelModel.from_pretrained(
    model_name="nvidia/speakerverification_en_titanet_large"
).to(device)

speaker_model.eval()

/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:numexpr.utils:NumExpr defaulting to 12 threads.
[NeMo W 2026-02-16 12:30:36 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.
W0216 12:30:36.914000 77551 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
INFO:nv_one_logger.exporter.export_config_manager:Final configuration contains 0 exporter(s)
[NeMo W 2026-02-16 12:30:38 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_libris

device: mps
[NeMo I 2026-02-16 12:30:38 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-16 12:30:38 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /Users/anshulchiranth/.cache/huggingface/hub/models--nvidia--speakerverification_en_titanet_large/snapshots/0dc382f40121a5fbd34db10a2bb04d826c2be6a8/speakerverification_en_titanet_large.nemo.


EncDecSpeakerLabelModel(
  (loss): AngularSoftmaxLoss()
  (eval_loss): AngularSoftmaxLoss()
  (_accuracy): TopKClassificationAccuracy()
  (preprocessor): AudioToMelSpectrogramPreprocessor(
    (featurizer): FilterbankFeatures()
  )
  (encoder): ConvASREncoder(
    (encoder): Sequential(
      (0): JasperBlock(
        (mconv): ModuleList(
          (0): MaskedConv1d(
            (conv): Conv1d(80, 80, kernel_size=(3,), stride=(1,), padding=(1,), groups=80, bias=False)
          )
          (1): MaskedConv1d(
            (conv): Conv1d(80, 1024, kernel_size=(1,), stride=(1,), bias=False)
          )
          (2): BatchNorm1d(1024, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (3): SqueezeExcite(
            (fc): Sequential(
              (0): Linear(in_features=1024, out_features=128, bias=False)
              (1): ReLU(inplace=True)
              (2): Linear(in_features=128, out_features=1024, bias=False)
            )
            (gap): AdaptiveAvgPool1d(

In [9]:
#Process for Embedding Ground Truth Embeddings
#They are saved to Ground Truth Embeddings/

emb_dir = base_dir / "Ground Truth Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in output_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(output_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

WAV files found: 2
example: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/To Be Embedded Ground Truths/Processed_George Robbins_converted.wav


100%|██████████| 2/2 [00:01<00:00,  1.26it/s]

Saved embeddings to: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Ground Truth Embeddings
                                            wav_path  \
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   

                                      embedding_path  dim  
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  


In [10]:
#Process for Embedding Transaction
#Saved to Transaction Embeddings/
emb_dir = base_dir / "Transaction Embeddings"
emb_dir.mkdir(parents = True, exist_ok = True)

wav_files = sorted([p for p in to_embed_dir.rglob("*.wav")])
print("WAV files found:", len(wav_files))
print("example:", wav_files[0] if wav_files else "None")

import numpy as np
import pandas as pd
from tqdm import tqdm

rows = []
all_embs = {}  # {relative_path: embedding_vector}

with torch.no_grad():
    for wav_path in tqdm(wav_files):
        # NeMo provides get_embedding(path_to_wav) for speaker embeddings :contentReference[oaicite:3]{index=3}
        emb = speaker_model.get_embedding(str(wav_path))

        # Normalize output type -> 1D numpy array
        if isinstance(emb, torch.Tensor):
            emb_np = emb.detach().cpu().numpy()
        else:
            emb_np = np.asarray(emb)

        emb_np = np.squeeze(emb_np)  # ensure shape (D,)

        # Save one embedding per file
        rel = wav_path.relative_to(to_embed_dir)
        out_path = emb_dir / rel.with_suffix(".npy")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, emb_np)

        rows.append({
            "wav_path": str(wav_path),
            "embedding_path": str(out_path),
            "dim": int(emb_np.shape[0]),
        })
        all_embs[str(rel)] = emb_np

df = pd.DataFrame(rows)
df.to_csv(emb_dir / "index.csv", index=False)

# Also save a single archive for easy loading later
np.savez_compressed(emb_dir / "embeddings.npz", **all_embs)

print("Saved embeddings to:", emb_dir)
print(df.head())

WAV files found: 67
example: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/To Be Embedded Transactions/George_10_converted.wav


100%|██████████| 67/67 [00:05<00:00, 11.86it/s]


Saved embeddings to: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/Transaction Embeddings
                                            wav_path  \
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...   

                                      embedding_path  dim  
0  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
1  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
2  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
3  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  
4  /Users/anshulchiranth/Desktop/Strike/Voice Exp...  192  


In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths (relative to current working directory) ---
tx_dir = Path.cwd() / "Transaction Embeddings"
gt_dir = Path.cwd() / "Ground Truth Embeddings"

# --- collect only .npy files, ignore everything else ---
tx_files = sorted([p for p in tx_dir.iterdir() if p.is_file() and p.suffix == ".npy"])
gt_files = sorted([p for p in gt_dir.iterdir() if p.is_file() and p.suffix == ".npy"])

if not tx_files:
    raise FileNotFoundError(f"No .npy files found in: {tx_dir}")
if not gt_files:
    raise FileNotFoundError(f"No .npy files found in: {gt_dir}")

# --- load helpers ---
def load_vec_192(path: Path) -> np.ndarray:
    """Load an embedding and return shape (192,) float32."""
    v = np.load(path)
    v = np.asarray(v, dtype=np.float32).reshape(-1)  # flatten any (1,192) or (192,1) etc.
    if v.shape[0] != 192:
        raise ValueError(f"{path.name}: expected 192 values, got {v.shape[0]} with original shape {np.load(path).shape}")
    return v

# --- load all embeddings ---
tx_mat = np.stack([load_vec_192(p) for p in tx_files], axis=0)   # (N, 192)
gt_mat = np.stack([load_vec_192(p) for p in gt_files], axis=0)   # (M, 192)

# --- cosine similarity (manual; no sklearn needed) ---
# sim = (A @ B.T) / (||A|| * ||B||)
tx_norm = np.linalg.norm(tx_mat, axis=1, keepdims=True)          # (N, 1)
gt_norm = np.linalg.norm(gt_mat, axis=1, keepdims=True)          # (M, 1)

# avoid divide-by-zero just in case
tx_norm = np.clip(tx_norm, 1e-12, None)
gt_norm = np.clip(gt_norm, 1e-12, None)

sim = (tx_mat @ gt_mat.T) / (tx_norm * gt_norm.T)               # (N, M)

# --- build dataframe ---
row_names = [p.stem for p in tx_files]
col_names = ["Cosine Similarity (George)", "Cosine Similarity (Julian)"]

df = pd.DataFrame(sim, index=row_names, columns=col_names).round(5)

# mapping from full column name → operator name
operator_map = {
    "Cosine Similarity (George)": "George",
    "Cosine Similarity (Julian)": "Julian"
}

# Get column name of max similarity per row
max_cols = df.idxmax(axis=1)

# Map to clean operator names
df["Recognized Operator"] = max_cols.map(operator_map)

# Convert index to a Series so we can use .str methods
index_series = df.index.to_series()

df["Ground Truth Operator"] = None  # initialize column

df.loc[index_series.str.contains("George", case=False, na=False),
       "Ground Truth Operator"] = "George"

df.loc[index_series.str.contains("Julian", case=False, na=False),
       "Ground Truth Operator"] = "Julian"

df["Correct Identification"] = (
    df["Recognized Operator"] == df["Ground Truth Operator"]
)


In [12]:
df

,Cosine Similarity (George),Cosine Similarity (Julian),Recognized Operator,Ground Truth Operator,Correct Identification
George_10_converted,0.64288,0.14007,George,George,True
George_11_converted,0.65613,0.13996,George,George,True
George_12_converted,0.76247,0.22466,George,George,True
George_13_converted,0.65088,0.02902,George,George,True
George_14_converted,0.44270,-0.09724,George,George,True
...,...,...,...,...,...
Julian_5_converted,-0.00239,0.70930,Julian,Julian,True
Julian_6_converted,-0.03831,0.65540,Julian,Julian,True
Julian_7_converted,-0.08511,0.65005,Julian,Julian,True
Julian_8_converted,0.00833,0.65165,Julian,Julian,True


In [13]:
df[df["Correct Identification"] == True]

,Cosine Similarity (George),Cosine Similarity (Julian),Recognized Operator,Ground Truth Operator,Correct Identification
George_10_converted,0.64288,0.14007,George,George,True
George_11_converted,0.65613,0.13996,George,George,True
George_12_converted,0.76247,0.22466,George,George,True
George_13_converted,0.65088,0.02902,George,George,True
George_14_converted,0.44270,-0.09724,George,George,True
...,...,...,...,...,...
Julian_5_converted,-0.00239,0.70930,Julian,Julian,True
Julian_6_converted,-0.03831,0.65540,Julian,Julian,True
Julian_7_converted,-0.08511,0.65005,Julian,Julian,True
Julian_8_converted,0.00833,0.65165,Julian,Julian,True


In [14]:
accuracy = (len(df[df["Correct Identification"] == True]) / len(df) ) * 100
print(f"Final Accuracy for binary classification task is: {accuracy}%")

Final Accuracy for binary classification task is: 100.0%
